# Budget and Policy

`Budget` and `Policy` control cost and PII handling during normalization. Defaults are deterministic and safe.

- **Budget:** `max_total_ms`, `max_per_field_ms`, `max_concurrency` — caps on execution time.
- **Policy:** `log_raw_input`, `currency_policy`, `unknown_handling` — privacy and behavior controls.

In [ ]:
from decimal import Decimal
from pydantic import BaseModel

import paxman
import paxman.contract.adapters.pydantic  # noqa: F401


class Product(BaseModel):
    name: str
    price: Decimal
    sku: str


# First, show the defaults
print("Default Budget:", paxman.Budget())
print("Default Policy:", paxman.Policy())
print()

In [ ]:
# Run with a tiny budget that forces UNRESOLVED for some fields
sample = "Widget A, price $12.99, SKU: WDG-001"

tight = paxman.normalize(
    sample,
    Product,
    budget=paxman.Budget(max_total_ms=1),  # unrealistically tight
    policy=paxman.Policy(log_raw_input=False),
)
print(f"Status: {tight.status.name}")
print(f"Unresolved fields: {tight.unresolved_fields}")
print(f"Diagnostics: {tight.diagnostics[:3]}..." if len(tight.diagnostics) > 3 else f"Diagnostics: {tight.diagnostics}")
print()

# Now run with default budget
default = paxman.normalize(sample, Product, policy=paxman.Policy(log_raw_input=False))
print(f"Default status: {default.status.name}")
print(f"Default unresolved: {default.unresolved_fields}")
print(f"Default data: {default.normalized_data}")

In [ ]:
# Determinism: same budget + same contract = same hash
run1 = paxman.normalize("Widget A, price $12.99", Product, policy=paxman.Policy(log_raw_input=False))
run2 = paxman.normalize("Widget A, price $12.99", Product, policy=paxman.Policy(log_raw_input=False))
print(f"run1 hash: {run1.replay_hash}")
print(f"run2 hash: {run2.replay_hash}")
print(f"Deterministic: {run1.replay_hash == run2.replay_hash}")

## Key settings

**Budget fields:**
- `max_total_ms` — total wall-clock time for normalization (default: no limit)
- `max_per_field_ms` — time per individual field (default: no limit)

**Policy fields:**
- `log_raw_input` — set to `False` (default) to prevent raw PII appearing in logs
- `currency_policy` — how to handle unknown or conflicting currencies
- `unknown_handling` — how to treat fields not declared in the contract

> **Tip:** Always set `log_raw_input=False` in production to avoid leaking PII into structured logs.

## Try it yourself

- Compare artifact hashes across different `Budget` values — they should differ because the execution path changes.
- Set `log_raw_input=True` and inspect the logs from `make playground-logs`.
- What happens when `max_per_field_ms=0`?